# Transform Data for REDCap Import 
V01.0 (2026-02-19)<br>
by Michael Rabenstein (__[ORCID](https://orcid.org/0000-0001-7712-224X)__)<br>
Code written with the help of: Perplexity AI. (2025). Perplexity AI: AI-powered search engine. https://www.perplexity.ai<br>
and<br>
UKB‑GPT (2026-02-17; institutional LLM based on openai/gpt-oss-120b (OpenAI et al. (2025). gpt-oss-120b & gpt-oss-20b Model Card. arXiv. https://doi.org/10.48550/arXiv.2508.10925))<br>
### Manual
This script is intended to facilitate the preparation of data tables for the REDCap import.<br>
In the script, the data in the import file of a project is compared with the parameters stored in the data dictionary and transformed if possible.<br> 
**Input:** Data table with the field names of the project in the header and the data dictionary.<br>
**Output:** Transformed data table with values in text format; a log file with a list of incorrect values.<br>
**Transform_Data_for_REDCap_Import_Methods.py** Contains functions, used by the notebook. adapt them, in case other validations rules are defined in your REDCap project.
**Block “Set variables and Paths; Load packages”<.** The paths to the two files are set manually here. In addition, the existing date format of the data table and the delimiters used for the input and output files are set. The block also checks whether the file paths are correct.<br>
**Block "Import data table and dictionary":** Import the data table and dictionary.<br>
**Block “Execute transformation”:** The data points are checked and transformed here. If a value cannot be transformed, e.g. text in a number field, the value is replaced by “errorVal”. If numerical values are outside the min-max range defined in the dictionary, they are logged with the field name, value and index. However, these values are not replaced by “errorVal” as REDCap accepts these values.<br> 
If the record ID column is missing in the data table, it is added and filled with values in ascending order from 1.
Errors during the date transformation, the number of “errorVal”, the values outside the min-max ranges and the header of the transformed data table are output in the block.<br>
**Block “Export transformed datatable and log”:** Saves the transformed datatable and a log file with the number of “errorVal” and the values outside the min-max ranges. These are saved as a CSV in the data table directory in the “Export” subfolder. The file names consist of the name of the data table, “_transformed_”, and the current datetime. The log file is also extended by “_log”. This means that exported data tables are not overwritten if the export is carried out several times.<br>
<br>
**Manual rework:** You can search for “errorVal” in the exported table. The position determined in this way can then be used in the original data table to check the incorrect values.<br>
<br>
## Set variables and Paths; Load packages

In [ ]:
# Install a pip package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install "numpy==2.4.3"
!{sys.executable} -m pip install "pandas==3.0.2"

It contains the code for the generation of the dummy data (Create-Dummy-Data-for-TDfRI_01.ipynb), merging different sheets in an Excel-spreadsheet to a single CSV (Combine-Excel-Sheets-to-CSV_05.R)

In [ ]:
import os
print('Use this path to set datatable_path and data_dictionary_path:')
print(os.getcwd())

In [ ]:
# Set path to files manually
# datatable_path = '/home/jovyan/transform-data-for-redcap-import/05_TransformDataforREDCapImport_Prepared-for-validation-check.csv'
datatable_path = '/home/jovyan/transform-data-for-redcap-import/07_TransformDataforREDCapImport_Prepared-for-validation-check.csv'

data_dictionary_path = '/home/jovyan/transform-data-for-redcap-import/TransformDataforREDCapImportTe_DataDictionary_2026-04-09.csv'

#dateformat_source = '%d-%m-%Y'
#dateformat_source = '%d.%m.%Y' #German format
dateformat_source = '%Y-%m-%d'
delimiter_datatable_source = ';' #Excel
#delimiter_datatable_source = ',' #OpenRefine
delimiter_dictionary = ','
delimiter_datatable_output = ';'
data_identifier_column = 'pseudonym'
recordid_prefix = 'Import_'

In [ ]:
# Supported file extensions
supported_datatable_exts = ['.csv', '.tsv', '.txt', '.xls', '.xlsx']
supported_dictionary_exts = ['.csv']

#import tkinter as tk
#from tkinter import filedialog, messagebox
import sys
import os
import pandas as pd
import numpy as np
from datetime import datetime
from zoneinfo import ZoneInfo
#import re
#import traceback
import Transform_Data_for_REDCap_Import_Methods as locmet

#Check if datatable_path and data_dictionary_path are valid
def check_file(path, supported_exts, description):
    _, ext = os.path.splitext(path)
    if ext.lower() not in supported_exts:
        print(f"{description}: File type '{ext}' is not supported (Path: {path})")
    elif not os.path.exists(path):
        print(f"{description}: File does not exist (Path: {path})")

check_file(datatable_path, supported_datatable_exts, "Datatable")
check_file(data_dictionary_path, supported_dictionary_exts, "Data Dictionary")

## Import data table and dictionary

In [ ]:
# Read all columns as text (object)
datatable = locmet.read_datatable(datatable_path, delimiter_datatable_source)
datatable_ori = datatable.copy()
data_dictionary = pd.read_csv(data_dictionary_path, dtype=str, delimiter = delimiter_dictionary) #, delimiter = delimiter_dictionary)

# Extract variable information
variable_names = data_dictionary.iloc[:, 0]
field_types    = data_dictionary.iloc[:, 3]
choices_list   = data_dictionary.iloc[:, 5]
formats        = data_dictionary.iloc[:, 7]
min_values     = data_dictionary.iloc[:, 8]
max_values     = data_dictionary.iloc[:, 9]

#print(formats)

## Execute transformation

In [ ]:
# Assume variable_names is a pandas Series or list with all variable names from the dictionary
first_var_name = variable_names.iloc[0] if hasattr(variable_names, 'iloc') else variable_names[0]
num_rows = len(datatable)

# Check if the first column header matches the first entry of variable_names
if datatable.columns[0] != first_var_name:
    # The first column is missing as header: add a new column at the beginning
    print(f"First column is not '{first_var_name}'. Adding new column with this name.")
    # Create ascending strings from 1 to num_rows
    new_col = [recordid_prefix + str(i) for i in range(1, num_rows + 1)]
    # Insert the new column at the beginning
    datatable.insert(0, first_var_name, new_col)
else:
    # The column is present
    col = datatable[first_var_name]
    # Check if it is completely empty (all values missing/NaN/empty)
    if col.isnull().all() or (col.astype(str).str.strip() == '').all():
        print(f"Column '{first_var_name}' is empty. Filling with ascending strings.")
        datatable[first_var_name] = [recordid_prefix + str(i) for i in range(1, num_rows + 1)]
    else:
        # Check if individual values are missing and fill them with 'errorVal'
        mask_missing = col.isnull() | (col.astype(str).str.strip() == '')
        if mask_missing.any():
            print(f"Column '{first_var_name}' has missing entries. Filling them with 'errorVal'.")
            datatable.loc[mask_missing, first_var_name] = 'errorVal'

# Initialize log for out-of-range values
out_of_range_log = []

# Main loop: process each variable according to its field_type
for var_name, format_str, field_type, choices_str, min_val, max_val in zip(
        variable_names, formats, field_types, choices_list, min_values, max_values):
    #format_encode = [line[1] for line in format_encode_list if line[0] == format_str]
    #print(format_encode)
    #print(format_str)
    if var_name in datatable.columns or (
        field_type == 'checkbox' and any(col.startswith(var_name + '___') for col in datatable.columns)
    ):
        locmet.process_column_by_field_type(
            datatable, var_name, format_str, field_type, choices_str, min_val, max_val, out_of_range_log, dateformat_source
        )

# Replace all NaN in the table with an empty string
datatable = datatable.fillna("")
datatable = datatable.replace("emptyValue", "")

# Count errorVal occurrences
num_errorVal = (datatable == "errorVal").sum().sum()

# Show results
print("Number of errorVal:", num_errorVal)
print("Values out of min/max range (log):")
log_df = pd.DataFrame(out_of_range_log)
print(log_df)
#print(datatable.head())
pd.set_option('display.max_columns', None)
datatable

## Export transformed datatable, log and error report

In [ ]:
# Prepare export file paths and names

# Get current datetime for filenames
#now_str = datetime.now().strftime("%Y%m%d%H%M%S")
now_str = datetime.now(ZoneInfo("Europe/Berlin")).strftime("%Y%m%d_%H%M%S")
# Get directory and base filename (without extension) from datatable_path
datatable_dir = os.path.dirname(datatable_path)
datatable_base = os.path.splitext(os.path.basename(datatable_path))[0]

# Ensure 'Export' subfolder exists
export_dir = os.path.join(datatable_dir, 'Export')
os.makedirs(export_dir, exist_ok=True)

# Build output filenames
datatable_export_name = f"{datatable_base}_transformed_{now_str}.csv"
datatable_export_path = os.path.join(export_dir, datatable_export_name)

log_export_name = f"{datatable_base}_transformed_{now_str}_log.csv"
log_export_path = os.path.join(export_dir, log_export_name)

error_export_name = f"{datatable_base}_transformed_{now_str}_errors.csv"
error_export_path = os.path.join(export_dir, error_export_name)

# Count 'errorVal' in datatable as integer
num_errorVal = str(int((datatable == 'errorVal').sum().sum()))

# Export datatable as CSV
#datatable.to_csv(datatable_export_path, sep=delimiter_datatable_output, index=False)
datatable.to_csv(datatable_export_path, sep=delimiter_datatable_output, index=False, encoding="utf-8-sig") #Umlaut-Rescue
print(f"Exported transformed datatable to: {datatable_export_path}")

# 4. Prepare and export log
log_df = pd.DataFrame(out_of_range_log)

base_cols = ["record_id", data_identifier_column]          # Names of your ID columns
report_df = datatable[base_cols].copy()                  # Copy from the transformed DataFrame

# Select exactly the lines that appear in the log
# `log_df[‘index’]` contains the original row indices (integer positions)
# We use these to retrieve only the corresponding rows from `log_report_df`.
# If the log DataFrame no longer has an “index” column (e.g., after renaming),
# it may need to be created first (see the methods section above).

#   - `iloc` uses integer-based indexing, which is exactly what
#     `log_df[‘index’]` returns.
#   - `reset_index(drop=True)` prevents the old index from being carried over
#     as a new column.
selected_rows = report_df.iloc[log_df["index"]].reset_index(drop=True)

# 4.3  Append the two columns to the LogDataFrame
log_df = pd.concat([log_df, selected_rows], axis=1)

# Add the errorVal count as a first row (optional: as a comment or as a separate file)
# Here, we add a column with the errorVal count for clarity
# If you want it as a comment line, you can write it manually before the DataFrame

# Option 1: Add as a separate row (with empty columns for the DataFrame)
log_header = pd.DataFrame([{
    'Number of errorVal': num_errorVal,
    'variable': '',
    'value': '',
    'index': '',
    'min value': '',
    'max value': '',
    'record_id': '',
    data_identifier_column: ''
}])
log_export_df = pd.concat([log_header, log_df], ignore_index=True)

cols = list(log_export_df.columns)

# If the “index” column appears, remove it and append it to the end
if "index" in cols:
    cols.remove("index")          # remove from the center
    cols.append("index")          # append to the end

# Rearrange the DataFrame in a new order
log_export_df = log_export_df[cols]

# Option 2: If you want only the DataFrame, just export log_df
#log_export_df.to_csv(log_export_path, sep=delimiter_datatable_output, index=False)
log_export_df.to_csv(log_export_path, sep=delimiter_datatable_output, index=False, encoding="utf-8") #Umlaut-Rescue
print(f"Exported log to: {log_export_path}")

# Find all columns that contain at least one “errorVal”
error_cols = [
    col for col in datatable.columns
    if (datatable[col] == "errorVal").any()
]

# For each column with errors, add the values from the transformed and original DataFrames
for col in error_cols:
    # Corresponding original column (from the original DataFrame)
    # We add the suffix “_ori” so that both columns in the report
    # can be uniquely identified.
    orig_col_name = f"{col}_ori"
    report_df[orig_col_name] = datatable_ori[col]
    
    # Column containing the “errorVal” entries (from the transformed DataFrame)
    report_df[col] = datatable[col]

# Export the error report
report_df.to_csv(error_export_path, sep=delimiter_datatable_output, index=False, encoding="utf-8") #Umlaut-Rescue
print(f"Exported error report to: {error_export_path}")

# 5. (Optional) Print summary to console as before
print("Number of errorVal:", num_errorVal)
print("Values out of min/max range (log):")
print(log_df)